In [1]:
%load_ext manim

The manim module is not an IPython extension.


In [2]:
from graph_animation.graph import Graph

In [3]:
# Execute the next cell to render ContainerAnimation.

from pathlib import Path

from manim import *
import numpy as np

from graph_animation import Graph
from graph_animation.primitives import ContainerBlob


def save_frame(scene, number):
    """Save the current scene image in a stable, numbered location."""
    output_dir = Path("frames_containers_animation")
    output_dir.mkdir(exist_ok=True)
    scene.renderer.update_frame(scene)
    scene.camera.get_image().save(output_dir / f"{number:02d}.png")


class ContainerAnimation(Scene):
    def construct(self):
        self.camera.background_color = WHITE

        positions = {
            "a": np.array([-4.2,  1.3, 0]), "b": np.array([-2.2,  2.0, 0]),
            "c": np.array([ 0.3,  1.5, 0]), "d": np.array([-3.0,  0.0, 0]),
            "e": np.array([-0.5,  0.4, 0]), "f": np.array([ 1.2, -0.2, 0]),
            "g": np.array([-4.1, -1.5, 0]), "h": np.array([-1.8, -1.5, 0]),
            "i": np.array([ 0.3, -1.4, 0]),
        }
        edges = [
            ("a", "b"), ("a", "d"), ("b", "c"), ("b", "d"),
            ("b", "e"), ("c", "e"), ("c", "f"), ("d", "e"),
            ("d", "g"), ("d", "h"), ("e", "f"), ("e", "h"),
            ("f", "i"), ("h", "i"),
        ]
        graph = Graph(list(positions), edges, positions)

        title = Text("", font_size=30, color=BLACK).to_edge(UP)

        def fingerprint_label(index, values):
            contents = r"\varnothing" if not values else r"\{" + ",".join(values) + r"\}"
            return MathTex(rf"F_{index}=" + contents, color=BLACK).to_edge(UP)

        def container_label(index, values):
            contents = r"\varnothing" if not values else r"\{" + ",".join(values) + r"\}"
            return MathTex(rf"C_{index}=" + contents, color=GREEN_D).scale(1.0).to_edge(DOWN)

        all_vertices = list(positions)
        fingerprint_tex = fingerprint_label(0, [])
        current_container_label = container_label(0, all_vertices)
        # Start with the conventional outline for the full vertex set.
        container = ContainerBlob(
            [graph[name] for name in all_vertices],
            padding=0.30,
            color=GREEN_D,
            shape="convex",
        )

        self.play(FadeIn(title), FadeIn(graph.view), run_time=0.8)
        self.play(
            Create(container),
            Write(fingerprint_tex),
            Write(current_container_label),
            graph.show_degrees(),
            run_time=0.8,
        )
        self.bring_to_back(container)

        frame = 1
        save_frame(self, frame)  # F_0 and C_0
        frame += 1

        # F_1 = {c}.  Delete vertices whose degree is larger than deg(c),
        # pause, then delete c and its still-active neighbours.
        next_fingerprint = fingerprint_label(1, ["c"])
        self.play(graph.select("c"), ReplacementTransform(fingerprint_tex, next_fingerprint))
        fingerprint_tex = next_fingerprint
        save_frame(self, frame)
        frame += 1

        high_degree = [
            name for name in graph
            if name != "c" and graph.is_active(name) and graph.degree(name) > graph.degree("c")
        ]
        for v in high_degree:
            print(v)
        
        self.play(
            graph.gray_vertices(high_degree),
            graph.update_degrees(),
            run_time=0.7,
        )
        self.wait(1.0)
        save_frame(self, frame)  # after the higher-degree deletion
        frame += 1

        c_neighbours = [name for name in graph.neighbors("c") if graph.is_active(name)]
        c_container = ["a", "g", "h", "i"]
        next_container_label = container_label(1, c_container)
        self.play(
            graph.gray_vertices(["c", *c_neighbours]),
            graph.update_degrees(),
            run_time=1.0,
        )
        save_frame(self, frame)  # C_1 = {a,g,h,i}
        frame += 1
        
        self.play(
            container.animate_to(
                [graph[name] for name in c_container], shape="connected"
            ),
            ReplacementTransform(current_container_label, next_container_label),
            run_time=1.0,
        )
        current_container_label = next_container_label
        save_frame(self, frame)  # C_1 = {a,g,h,i}
        frame += 1

        # F_2 = {c,h}; h and its remaining neighbour i leave C_2 = {a,g}.
        next_fingerprint = fingerprint_label(2, ["c", "h"])
        self.play(graph.select("h"), ReplacementTransform(fingerprint_tex, next_fingerprint))
        fingerprint_tex = next_fingerprint
        save_frame(self, frame)
        frame += 1

        h_neighbours = [name for name in graph.neighbors("h") if graph.is_active(name)]
        h_container = ["a", "g"]
        next_container_label = container_label(2, h_container)
        self.play(
            graph.gray_vertices(["h", *h_neighbours]),
            graph.update_degrees(),
            container.animate_to([graph[name] for name in h_container]),
            ReplacementTransform(current_container_label, next_container_label),
            run_time=1.0,
        )
        current_container_label = next_container_label
        save_frame(self, frame)  # C_2 = {a,g}
        frame += 1

        # F_3 = {c,h,a}; the terminal isolated vertex is removed with a.
        next_fingerprint = fingerprint_label(3, ["c", "h", "a"])
        self.play(graph.select("a"), ReplacementTransform(fingerprint_tex, next_fingerprint))
        fingerprint_tex = next_fingerprint
        save_frame(self, frame)
        frame += 1

        final_container = container_label(3, [])
        remaining = [name for name in graph if graph.is_active(name)]
        self.play(
            graph.gray_vertices(remaining),
            container.animate_to([]),
            ReplacementTransform(current_container_label, final_container),
            run_time=0.9,
        )
        save_frame(self, frame)  # C_3 = empty
        self.wait(1)


In [5]:
%%manim -v WARNING -qh --disable_caching ContainerAnimation

# Run the preceding setup cell first; it defines ContainerAnimation.


Manim Community v0.20.1

b
d
e
